<a href="https://colab.research.google.com/github/Amulya-Rao-mj/GENAI-lab/blob/main/GENAI_10_Integrated_genai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq gradio sentence-transformers faiss-cpu pypdf

In [3]:
import os
from google.colab import userdata

# Automatically fetches your saved key from Colab Secrets
os.environ["GENAI_KEY"] = userdata.get("GENAI_KEY")

In [4]:
import os
import json
import faiss
import numpy as np
import gradio as gr
from pypdf import PdfReader
from groq import Groq
from sentence_transformers import SentenceTransformer

# 1. Initialize Groq Client & Local Embedding Model
client = Groq(api_key=os.environ.get("GENAI_KEY"))
MODEL_NAME = "llama-3.3-70b-versatile"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Global state for our in-memory Vector Store
vector_index = None
document_chunks = []

# ==========================================
# 2. RAG PIPELINE (Ingestion & Retrieval)
# ==========================================
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def process_files(files):
    global vector_index, document_chunks
    if not files:
        return "⚠️ No files uploaded."

    all_text = ""
    for file in files:
        if file.name.endswith(".pdf"):
            reader = PdfReader(file.name)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    all_text += extracted + "\n"
        elif file.name.endswith(".txt") or file.name.endswith(".md"):
            with open(file.name, "r", encoding="utf-8") as f:
                all_text += f.read() + "\n"

    document_chunks = chunk_text(all_text)
    if not document_chunks:
        return "⚠️ Could not extract any readable text from the uploaded files."

    # Generate embeddings & build FAISS index
    embeddings = embedder.encode(document_chunks)
    dimension = embeddings.shape[1]
    vector_index = faiss.IndexFlatL2(dimension)
    vector_index.add(np.array(embeddings).astype("float32"))

    return f"✅ Indexed {len(document_chunks)} chunks from {len(files)} file(s) into FAISS!"

def search_knowledge_base(query: str, top_k: int = 3) -> str:
    """Tool function: Searches uploaded documents via FAISS vector similarity."""
    if vector_index is None or len(document_chunks) == 0:
        return "Knowledge base is currently empty. Ask the user to upload documents first."

    query_vec = embedder.encode([query]).astype("float32")
    distances, indices = vector_index.search(query_vec, top_k)

    results = []
    for idx in indices[0]:
        if idx < len(document_chunks):
            results.append(document_chunks[idx])

    return "\n---\n".join(results)

# ==========================================
# 3. AGENT TOOLS & ORCHESTRATION LOOP
# ==========================================
def calculate_math(expression: str) -> str:
    """Tool function: Safely evaluates basic math expressions."""
    try:
        allowed_chars = set("0123456789+-*/().% ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Unsupported characters in math expression."
        return str(eval(expression))
    except Exception as e:
        return f"Calculation error: {str(e)}"

# Define tool schemas for Groq's function calling API
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search the user's uploaded PDF or text documents for facts, context, or specific details.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up in the documents."
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_math",
            "description": "Evaluate arithmetic or mathematical expressions accurately.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid math expression, e.g., '(1250 * 0.18) + 45'"
                    }
                },
                "required": ["expression"],
            },
        },
    },
]

AVAILABLE_FUNCTIONS = {
    "search_knowledge_base": search_knowledge_base,
    "calculate_math": calculate_math,
}

def run_agent(user_message, chat_history):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an intelligent AI agent equipped with RAG (knowledge base search) and a calculator. "
                "When the user asks about uploaded files, policies, or specific documents, ALWAYS call "
                "`search_knowledge_base`. When math is required, call `calculate_math`. "
                "Cite retrieved context clearly and concisely."
            ),
        }
    ]

    # Append prior conversation turns
    for user_turn, assistant_turn in chat_history:
        messages.append({"role": "user", "content": user_turn})
        if assistant_turn:
            messages.append({"role": "assistant", "content": assistant_turn})

    messages.append({"role": "user", "content": user_message})

    agent_trace = []

    # Step A: Initial LLM call (decides whether to use tools or respond directly)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        max_tokens=1024,
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # Step B: Execute any requested tools in an agentic loop
    if tool_calls:
        messages.append(response_message)

        for tool_call in tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            agent_trace.append(f"🛠️ **Tool Called:** `{fn_name}({fn_args})`")

            fn_to_call = AVAILABLE_FUNCTIONS.get(fn_name)
            tool_result = fn_to_call(**fn_args) if fn_to_call else "Tool not found."

            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": fn_name,
                    "content": tool_result,
                }
            )

        # Step C: Final synthesis call after receiving tool outputs
        final_response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            max_tokens=1024,
        )
        final_text = final_response.choices[0].message.content
    else:
        agent_trace.append("💬 **Direct LLM Response** (No tools needed)")
        final_text = response_message.content

    trace_display = "\n".join(agent_trace)
    formatted_reply = f"{final_text}\n\n---\n*{trace_display}*"
    return formatted_reply

# ==========================================
# 4. GRADIO INTERFACE
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚡ Groq Agentic RAG Assistant")
    gr.Markdown("Upload PDFs/TXT files to build your knowledge base, then ask questions or run calculations.")

    with gr.Row():
        with gr.Column(scale=1):
            file_upload = gr.File(
                label="Upload Knowledge Base (.pdf, .txt, .md)",
                file_count="multiple",
                file_types=[".pdf", ".txt", ".md"],
            )
            index_btn = gr.Button("Build Vector Index", variant="primary")
            status_box = gr.Textbox(label="RAG Index Status", interactive=False)

            index_btn.click(
                fn=process_files,
                inputs=[file_upload],
                outputs=[status_box],
            )

        with gr.Column(scale=2):
            gr.ChatInterface(
                fn=run_agent,
                examples=[
                    "Summarize the key points from my uploaded documents.",
                    "If revenue is 48500 and expenses are 19230, what is the profit margin percentage?",
                    "Explain how Retrieval-Augmented Generation works.",
                ],
            )

# share=True is required in Google Colab to generate a public tunnel URL
demo.launch(share=True, debug=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (7 outputs left out)

/tmp/ipykernel_834/3149681240.py:205: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ebdad6abd06d5fb1c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 1083, in async_wrapper
    response = await 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ebdad6abd06d5fb1c6.gradio.live
